Phase 3 — Accuracy Evaluation

Goal:
Evaluate Top-1 accuracy retention for:

1. Baseline FP32
2. Structural Pruning (P)
3. Structural Pruning → Quantization (P → Q)
4. Structural Quantization → Pruning (Q → P, repaired)

using an ImageNet-compatible validation dataset.

In [1]:
import torch
import torchvision
import torchvision.transforms as transforms

from torchvision.models import mobilenet_v2

In [2]:
print(torch.__version__)
print(torchvision.__version__)

2.11.0+cu126
0.26.0+cu126


In [3]:
from torchvision.datasets import ImageFolder

print("Ready")

Ready


In [4]:
import os

os.makedirs("../data", exist_ok=True)

print("Data folder ready.")

Data folder ready.


In [6]:
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torchvision.datasets.utils import download_and_extract_archive
import os

In [7]:
DATA_DIR = "../data"

url = "https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz"

download_and_extract_archive(
    url=url,
    download_root=DATA_DIR,
    remove_finished=False
)

print("Download complete.")

100.0%


Download complete.


In [8]:
import os
print(os.listdir("../data"))

['imagenette2-160', 'imagenette2-160.tgz']


In [9]:
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [10]:
val_dataset = ImageFolder(
    "../data/imagenette2-160/val",
    transform=transform
)

print("Validation images:", len(val_dataset))

Validation images: 3925


In [11]:
from torch.utils.data import DataLoader

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

print("Validation loader ready.")

Validation loader ready.


In [12]:
print(val_dataset.classes)

['n01440764', 'n02102040', 'n02979186', 'n03000684', 'n03028079', 'n03394916', 'n03417042', 'n03425413', 'n03445777', 'n03888257']


In [13]:
baseline_model = mobilenet_v2(weights="DEFAULT")
baseline_model.eval()

print("Baseline model loaded.")

Baseline model loaded.


In [14]:
def evaluate_accuracy(model, dataloader, device="cpu"):
    model.eval()
    model.to(device)

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return 100 * correct / total

In [15]:
baseline_accuracy = evaluate_accuracy(
    baseline_model,
    val_loader,
    "cpu"
)

print(f"Baseline Accuracy: {baseline_accuracy:.2f}%")

Baseline Accuracy: 9.12%
